# UAS Data Mining
## Analisis & Pelatihan Model LightGBM


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, json, warnings
warnings.filterwarnings('ignore')

# Install jika belum ada
# !pip install lightgbm
import lightgbm as lgb
print('Library OK — LightGBM', lgb.__version__)

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../dataset/data.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. EDA

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())
df.describe()

In [ ]:
# Distribusi target
TARGET_COL = 'label'   # <-- GANTI dengan nama kolom target
print(df[TARGET_COL].value_counts())
df[TARGET_COL].value_counts().plot(kind='bar', figsize=(6,3), color='steelblue')
plt.title('Distribusi Kelas Target'); plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

FEATURE_COLS = [c for c in df.columns if c != TARGET_COL]

# Encode kategorik jika ada
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    if col != TARGET_COL:
        df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

X = df_enc[FEATURE_COLS]
y = df_enc[TARGET_COL]
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 4. Pelatihan Model LightGBM

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

lgbm_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    verbose=-1
)
lgbm_model.fit(X_train, y_train)

y_pred = lgbm_model.predict(X_test)
print(f'Akurasi: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred))

## 5. Feature Importance

In [ ]:
import pandas as pd
fi = pd.DataFrame({'Fitur': FEATURE_COLS, 'Importance': lgbm_model.feature_importances_})
fi = fi.sort_values('Importance', ascending=False)
fi.head(15).plot(kind='barh', x='Fitur', y='Importance', figsize=(8,5), color='steelblue')
plt.gca().invert_yaxis(); plt.title('Feature Importance'); plt.tight_layout(); plt.show()

## 6. Simpan Model & Metadata

In [ ]:
# Simpan model
with open('../model/lgbm_model.pkl', 'wb') as f:
    pickle.dump(lgbm_model, f)

# Tipe data fitur
type_map = {'int64':'int','int32':'int','float64':'float','float32':'float','object':'category'}
feature_types = {col: type_map.get(str(df[col].dtype), 'float') for col in FEATURE_COLS}

# Label map
orig_classes = sorted(df[TARGET_COL].unique())
label_map    = {str(i): str(c) for i, c in enumerate(orig_classes)}

meta = {
    'features': FEATURE_COLS,
    'types': feature_types,
    'target': TARGET_COL,
    'label_map': label_map,
    'model_info': {'algorithm': 'LightGBM', 'version': '1.0'}
}

with open('../model/features.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Model disimpan ke: model/lgbm_model.pkl')
print('✅ Metadata disimpan ke: model/features.json')